# Seed Profiling
## Stage 1 of Dataset Discovery Pipeline

This notebook runs **Seed Profiling** on `data/seed.csv` and produces
`output/seed_signature.json` — a fully structured, machine-readable profile
used by the downstream Query Generation and Dataset Discovery stages.

---
### What this notebook does
1. Loads `data/seed.csv` via the `profiler.py` module
2. Runs column-level profiling (semantic types, stats, heuristics)
3. Extracts global signals (identifiers, entities, composite keys)
4. Scores quality (missingness, duplicates, outliers)
5. Infers concepts (time series, geospatial, economic, …)
6. Writes `output/seed_signature.json`
7. Displays an interactive summary of the result

> **Collaborators:** run all cells top-to-bottom (`Runtime → Run all`).  
> The only requirement is `pip install -r requirements.txt`.


## 1. Environment Setup

In [1]:
# Install dependencies (safe to re-run)
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install",
                       "-r", "../requirements.txt", "-q"])
print("Dependencies ready.")


Dependencies ready.


In [2]:
import sys, os
# Make src/ importable whether running from notebooks/ or root
for candidate in ["../src", "src"]:
    if os.path.isdir(candidate) and candidate not in sys.path:
        sys.path.insert(0, candidate)

from profiler import profile, save_signature
import pandas as pd
import json
from pathlib import Path
print("Imports OK.")


Imports OK.


## 2. Load the Seed Dataset

In [3]:
# Resolve path regardless of whether the notebook is opened from
# notebooks/ (Jupyter) or the project root (Colab).
_here = Path(".").resolve()
if (_here / "data" / "seed.csv").exists():
    SEED_PATH   = _here / "data" / "seed.csv"
    OUTPUT_PATH = _here / "output" / "seed_signature.json"
else:
    SEED_PATH   = _here.parent / "data" / "seed.csv"
    OUTPUT_PATH = _here.parent / "output" / "seed_signature.json"

df = pd.read_csv(SEED_PATH)
print(f"Loaded: {SEED_PATH}")
print(f"Shape : {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()


Loaded: C:\Users\kalle\Desktop\Seed-and-Seek\data\seed.csv
Shape : 32 rows × 14 columns


,country_code,country_name,year,gdp_usd,gdp_growth_pct,population,life_expectancy,infant_mortality_rate,literacy_rate_pct,urban_pop_pct,co2_emissions_kt,region,income_group,currency_code
0,USA,United States,2020,20936600000000,2.3,331002651,78.5,5.4,99.0,82.7,4712000,North America,High income,USD
1,USA,United States,2021,22996100000000,5.7,332915073,77.3,5.4,99.0,83.1,5007000,North America,High income,USD
2,USA,United States,2022,25462700000000,1.9,334805269,76.4,5.4,99.0,83.4,5200000,North America,High income,USD
3,DEU,Germany,2020,3846414000000,-4.6,83783942,81.3,3.1,99.0,77.5,644000,Europe,High income,EUR
4,DEU,Germany,2021,4259935000000,2.6,83900473,80.5,3.1,99.0,77.8,675000,Europe,High income,EUR


In [4]:
# Quick dtype + null overview
summary = pd.DataFrame({
    "dtype":      df.dtypes.astype(str),
    "null_count": df.isna().sum(),
    "null_pct":   (df.isna().mean() * 100).round(2),
    "unique":     df.nunique(),
})
summary


,dtype,null_count,null_pct,unique
country_code,str,1,3.12,11
country_name,str,0,0.00,11
year,int64,0,0.00,3
gdp_usd,int64,0,0.00,32
gdp_growth_pct,float64,0,0.00,31
population,int64,0,0.00,32
life_expectancy,float64,0,0.00,29
infant_mortality_rate,float64,0,0.00,11
literacy_rate_pct,float64,0,0.00,17
urban_pop_pct,float64,0,0.00,31


## 3. Run the Profiler

In [5]:
sig = profile(SEED_PATH)
save_signature(sig, OUTPUT_PATH)

print(f"\nDataset   : {sig['dataset_name']}")
print(f"Rows      : {sig['n_rows']:,}")
print(f"Columns   : {sig['n_columns']}")
print(f"Concepts  : {sig['inferred_concepts']}")
print(f"Output    : {OUTPUT_PATH}")


[profiler] Signature written to C:\Users\kalle\Desktop\Seed-and-Seek\output\seed_signature.json

Dataset   : seed.csv
Rows      : 32
Columns   : 14
Concepts  : ['demographics', 'economic', 'education', 'environmental', 'geospatial', 'public_health', 'time_series']
Output    : C:\Users\kalle\Desktop\Seed-and-Seek\output\seed_signature.json


## 4. Column-Level Profiles

Each column is annotated with:
- **semantic_type** — `numeric`, `datetime`, `categorical`, `identifier`, `entity`, `free_text`
- **null_ratio / unique_ratio** — coverage and cardinality signals
- **is_candidate_identifier** — likely a join key
- **is_useful_for_joining** — shared reference codes (geo, time, categorical)


In [6]:
col_df = pd.DataFrame([
    {
        "column":       c["name"],
        "dtype":        c["raw_dtype"],
        "semantic":     c["semantic_type"],
        "null_%":       f"{c['null_ratio']*100:.1f}" if c['null_ratio'] is not None else "-",
        "unique_%":     f"{c['unique_ratio']*100:.1f}" if c['unique_ratio'] is not None else "-",
        "identifier?":  "✓" if c["is_candidate_identifier"] else "",
        "entity?":      "✓" if c["is_entity_like"] else "",
        "query?":       "✓" if c["is_useful_for_querying"] else "",
        "join?":        "✓" if c["is_useful_for_joining"] else "",
    }
    for c in sig["columns"]
])
col_df


,column,dtype,semantic,null_%,unique_%,identifier?,entity?,query?,join?
0,country_code,str,identifier,3.1,34.4,✓,,✓,✓
1,country_name,str,categorical,0.0,34.4,,✓,✓,✓
2,year,int64,datetime,0.0,9.4,,,✓,✓
3,gdp_usd,int64,numeric,0.0,100.0,,,,
4,gdp_growth_pct,float64,numeric,0.0,96.9,,,,
5,population,int64,numeric,0.0,100.0,,,,✓
6,life_expectancy,float64,numeric,0.0,90.6,,,,
7,infant_mortality_rate,float64,numeric,0.0,34.4,,,,
8,literacy_rate_pct,float64,numeric,0.0,53.1,,,,
9,urban_pop_pct,float64,numeric,0.0,96.9,,,,


## 5. Global Extracted Signals

In [7]:
print("── Candidate identifiers ──────────────────")
for c in sig["candidate_identifiers"]:
    print(f"  {c}")

print("\n── Candidate composite keys ───────────────")
for pair in sig["candidate_composite_keys"]:
    print(f"  {' + '.join(pair)}")

print("\n── Entity-like columns ────────────────────")
for c in sig["entity_like_columns"]:
    print(f"  {c}")

print("\n── Important attributes (high-variance numeric) ──")
for c in sig["important_attributes"]:
    print(f"  {c}")


── Candidate identifiers ──────────────────
  country_code
  currency_code

── Candidate composite keys ───────────────

── Entity-like columns ────────────────────
  country_name
  region

── Important attributes (high-variance numeric) ──
  gdp_usd
  gdp_growth_pct
  population
  life_expectancy
  literacy_rate_pct
  urban_pop_pct
  co2_emissions_kt


## 6. Quality Profile

In [8]:
qp = sig["quality_profile"]

print(f"Duplicate rows     : {qp['duplicate_row_count']}")
print(f"High-missingness   : {qp['high_missingness_columns'] or 'none'}")

if qp["outlier_summary"]:
    print("\nOutliers (IQR 1.5x method):")
    for col, info in qp["outlier_summary"].items():
        print(f"  {col:30s}  {info['outlier_count']} outlier rows")
else:
    print("\nNo outliers detected.")


Duplicate rows     : 0
High-missingness   : none

Outliers (IQR 1.5x method):
  gdp_usd                         6 outlier rows
  gdp_growth_pct                  2 outlier rows
  population                      6 outlier rows
  infant_mortality_rate           3 outlier rows
  literacy_rate_pct               3 outlier rows
  co2_emissions_kt                3 outlier rows


## 7. Retrieval-Oriented Summary

This section is what the **Query Generation** stage will consume.
It contains salient column names, entity value samples, and inferred concepts.


In [9]:
print("── Salient column names ───────────────────────────────────")
print("  " + ", ".join(sig["salient_column_names"]))

print("\n── Entity value samples (seed for queries) ────────────────")
for col, vals in sig["entity_value_samples"].items():
    print(f"  {col}: {vals}")

print("\n── Inferred concepts ──────────────────────────────────────")
print("  " + ", ".join(sig["inferred_concepts"]))


── Salient column names ───────────────────────────────────
  country_code, country_name, year, population, region, income_group, currency_code

── Entity value samples (seed for queries) ────────────────
  country_name: ['United States', 'Germany', 'Brazil', 'India', 'Nigeria']
  region: ['North America', 'Europe', 'Latin America', 'South Asia', 'Sub-Saharan Africa']

── Inferred concepts ──────────────────────────────────────
  demographics, economic, education, environmental, geospatial, public_health, time_series


## 8. Full Signature JSON (preview)

In [10]:
# Print the full signature minus the verbose columns list for readability
preview = {k: v for k, v in sig.items() if k != "columns"}
print(json.dumps(preview, indent=2, default=str))


{
  "dataset_name": "seed.csv",
  "file_path": "C:\\Users\\kalle\\Desktop\\Seed-and-Seek\\data\\seed.csv",
  "file_size_bytes": 3585,
  "file_format": "CSV",
  "profiled_at": "2026-04-11T10:05:00.437802Z",
  "n_rows": 32,
  "n_columns": 14,
  "candidate_identifiers": [
    "country_code",
    "currency_code"
  ],
  "candidate_composite_keys": [],
  "entity_like_columns": [
    "country_name",
    "region"
  ],
  "important_attributes": [
    "gdp_usd",
    "gdp_growth_pct",
    "population",
    "life_expectancy",
    "literacy_rate_pct",
    "urban_pop_pct",
    "co2_emissions_kt"
  ],
  "quality_profile": {
    "duplicate_row_count": 0,
    "high_missingness_columns": [],
    "outlier_summary": {
      "gdp_usd": {
        "outlier_count": 6,
        "method": "IQR_1.5x"
      },
      "gdp_growth_pct": {
        "outlier_count": 2,
        "method": "IQR_1.5x"
      },
      "population": {
        "outlier_count": 6,
        "method": "IQR_1.5x"
      },
      "infant_mortality_rat

In [11]:
# Show the first 3 column profiles in full
for cp in sig["columns"][:3]:
    print(json.dumps(cp, indent=2, default=str))
    print()


{
  "name": "country_code",
  "raw_dtype": "str",
  "semantic_type": "identifier",
  "null_count": 1,
  "null_ratio": 0.0312,
  "unique_count": 11,
  "unique_ratio": 0.3438,
  "sample_values": [
    "USA",
    "DEU",
    "BRA",
    "IND",
    "NGA"
  ],
  "is_candidate_identifier": true,
  "is_entity_like": false,
  "is_useful_for_querying": true,
  "is_useful_for_joining": true
}

{
  "name": "country_name",
  "raw_dtype": "str",
  "semantic_type": "categorical",
  "null_count": 0,
  "null_ratio": 0.0,
  "unique_count": 11,
  "unique_ratio": 0.3438,
  "sample_values": [
    "United States",
    "Germany",
    "Brazil",
    "India",
    "Nigeria"
  ],
  "top_values": {
    "United States": 3,
    "Germany": 3,
    "Brazil": 3,
    "India": 3,
    "Nigeria": 3,
    "Japan": 3,
    "South Africa": 3,
    "Australia": 3,
    "Mexico": 3,
    "China": 3
  },
  "is_candidate_identifier": false,
  "is_entity_like": true,
  "is_useful_for_querying": true,
  "is_useful_for_joining": true
}

{


## 9. Output

The signature has been written to `output/seed_signature.json`.

```
output/
└── seed_signature.json   ← machine-readable profile for Query Generation
```

**Next step:** Query Generation will read this file and use
`salient_column_names`, `entity_value_samples`, and `inferred_concepts`
to construct discovery queries.
